# Lab 16: Regularization — Diagnosing a Coefficient You Cannot Interpret
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnostic Lab | 110 min Core + 25 min Extension

---

**Learning Objectives:**
- Measure how often a "95%" interval covers when LASSO chose the controls from the same data, and explain why the point estimate moves too
- Explain why a larger penalty makes that procedure worse, not better
- Implement Belloni-Chernozhukov-Hansen double selection and measure its coverage against OLS on the true controls
- Show that a LASSO coefficient is shrunk, and say why it cannot be read as an effect
- Choose λ by cross-validation with the scaler fitted inside every fold, in a loop you finish, and match `GridSearchCV` to every digit

**Dataset:** two simulated designs where every true coefficient is known: a treatment effect among 100 candidate controls, and a sparse design with 60 predictors

**Format:** Each diagnosis shows you a result that is wrong, or wrong-looking, and asks what happened. Write your answers in the text cell under each set of questions.

**Deliverable:** your written diagnoses (Diagnoses 1, 2 and 3), your `double_selection()` function from Diagnosis 2, the required `choose_lambda()` function, and `src/sparse_tools.py`, a module you finish with your `double_selection()`, save and upload.

**Time:** Core (110 min, including the required Write it yourself) + Extension (optional, 25 min)

With setup, the README and the GitHub submission, plan on about two and a quarter hours, most of it after class. The optional Extension adds about 25 minutes.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI-assisted coding section: every line you add is yours, including the required "Write it yourself" function. The one AI use is the README prompt at the end, which asks for documentation, not code.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Diagnosis 2:** a `double_selection()` function, the repair for Diagnosis 1, about four lines in an empty cell using the given `lasso_support()`, and one line under it that runs it on Diagnosis 1's sample. In Ship It you paste it, unchanged, into the module.
> - **Write it yourself (required, after Diagnosis 3):** finish `choose_lambda()`: its outer loop over λ is given, and you write the inner loop over the 5 folds, with the scaler fitted inside every fold, about four lines in place of a `____`. A check cell tells you when it is right. It is required.
> - **Extension (optional):** two sample-splitting estimators, from a three-step outline.
>
> **You write, in text cells:** three answers in Diagnosis 1, one in Diagnosis 2 and one in Diagnosis 3.
>
> **You run and read:** everything else. That is the simulated treatment effect, the analyst's LASSO-then-OLS code and its 200 samples, `lasso_support()`, the same 200 samples run through your `double_selection` beside OLS on the true controls, the shrinkage tables and the `GridSearchCV` choice of λ in Diagnosis 3, and the rest of Ship It's module, `sparse_tools.py`, which is working code to read, not to retype. Read each cell before you run it: the questions ask about what it prints, and Write it yourself rebuilds Diagnosis 3's `GridSearchCV` choice as a loop.

## Setup

The first line of the setup cell upgrades scikit-learn to 1.8 or later, because `LassoCV` changed
in 1.8 and the optional Extension's coverage comes from the newer version. The cell prints your
scikit-learn version. If it shows a version below 1.8, use **Runtime → Restart session** and run this cell again.

In [ ]:
# GIVEN — run as-is
# Setup: the libraries and the seed
!pip install "scikit-learn>=1.8" -q

from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import statsmodels.api as sm
from sklearn import metrics          # metrics and cross_val_score are for Write it yourself
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Path("src").mkdir(exist_ok=True)     # Ship It writes its module into this folder

RANDOM_STATE = 42                    # the seed for every LASSO and for the folds
print(f"scikit-learn {sklearn.__version__} (the Extension's coverage assumes 1.8 or later)")
print("Setup complete.")

## Diagnosis 1: The Interval That Ignores the Search (25 min)

An analyst has 200 observations: a treatment `d`, an outcome `y` and 100 candidate controls. She
runs LASSO to pick the controls, keeps the ones it did not set to zero, fits OLS of `y` on `d` and
those survivors, and reports OLS's standard error and 95% interval. This is the most common
high-dimensional mistake in applied work. The true effect of `d` is 1.0, and you know that because
you are about to simulate it.

**Reading the code.** `make(seed)` draws one sample. `r.normal(0, 1, (n, p))` draws a table of n
rows and p columns of standard normal numbers: the 100 candidate controls `Z`. `Z[:, :5]` is its
first five columns, and `@` is matrix multiplication, so `Z[:, :5] @ np.array([...])` adds up
those five columns, each times its number. The same five columns drive the treatment `d` and the
outcome `y`, so they are confounders; the other 95 controls do nothing. A function can hand back
several things at once, and `Z, d, y = make(...)` names them in order.

In [ ]:
# GIVEN — run as-is
# A treatment effect of 1.0, with five confounders among 100 candidate controls
TRUE_EFFECT = 1.0


def make(seed, n=200, p=100):
    """One sample: n rows of p candidate controls Z, a treatment d and an outcome y."""
    r = np.random.default_rng(seed)
    Z = r.normal(0, 1, (n, p))
    d = Z[:, :5] @ np.array([1.0, -0.8, 0.6, 0.5, -0.4]) + r.normal(0, 1, n)
    y = (TRUE_EFFECT * d + Z[:, :5] @ np.array([1.2, -1.0, 0.8, 0.6, -0.5])
         + r.normal(0, 1, n))
    return Z, d, y


Z, d, y = make(RANDOM_STATE)

**Reading the analyst's code.** `np.column_stack([d, Z])` puts `d` in front of the 100 controls
as column 0, so LASSO sees 101 columns. `LassoCV(cv=5, random_state=..., max_iter=20_000)` chooses
the penalty λ by 5-fold cross-validation, over 100 penalties of its own, with the rows dealt into
the folds in order; scikit-learn calls λ `alpha`. `max_iter` lets the solver take more passes
before it gives up, and `20_000` is 20000 (the underscore only helps you read it).

After `.fit(X, y)`, `.coef_` holds the coefficients. `[1:]` drops the first one, `d`'s, and
`np.abs(...) > 1e-8` is True for each control whose coefficient is not zero. `np.flatnonzero(...)`
gives the positions of the Trues, so `keep` lists the controls LASSO kept, and `Z[:, keep]` is
those columns. `sm.add_constant(X)` adds a column of ones, the intercept, so in `naive.params`
position 0 is the constant and position 1 is the coefficient on `d`. `.conf_int()[1]` is that
coefficient's 95% interval, two numbers that `lo, hi =` names in order. `Z.shape[1]` is the
number of columns of `Z`, and `{x:.4f}` prints x with 4 decimals.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# LASSO picks the controls, then OLS of y on d and the survivors reports the interval
las = LassoCV(cv=5, random_state=RANDOM_STATE, max_iter=20_000).fit(
    np.column_stack([d, Z]), y)
keep = np.flatnonzero(np.abs(las.coef_[1:]) > 1e-8)
naive = sm.OLS(y, sm.add_constant(np.column_stack([d, Z[:, keep]]))).fit()

lo, hi = naive.conf_int()[1]
print(f"controls selected : {len(keep)} of {Z.shape[1]}")
print(f"estimate          : {naive.params[1]:.4f}   truth {TRUE_EFFECT}")
print(f"95% interval      : [{lo:.4f}, {hi:.4f}]")

**Expected output:** 21 of the 100 controls selected, an estimate of 1.0495 against a truth of
1.0, and a 95% interval of [0.9272, 1.1717], which contains 1.0.

**One sample cannot tell you how often the interval covers; 200 can.** The next cell runs the
analyst's procedure on 200 new samples (seeds 4000 to 4199) and counts how often the interval
contains 1.0. `lo <= TRUE_EFFECT <= hi` is True or False, and adding True to a count adds 1.
`.append(x)` adds x to the end of a list. `keep_i < 5` is True for each kept control that is one
of the five confounders, columns 0 to 4, and `np.sum` counts the Trues. `fit.bse[1]` is the
standard error OLS reports for the coefficient on `d`, and `np.std(ests, ddof=1)` is the actual
spread of the 200 estimates (`ddof=1` makes it the sample standard deviation). The Monte Carlo
standard error of a share p measured on 200 samples is √(p(1 − p)/200), and `** 0.5` is the
square root. `{x:.1%}` prints a share as a percentage with one decimal. It takes about fifteen
seconds.

In [ ]:
# GIVEN — run as-is
# The analyst's procedure on 200 new samples: how often does the "95%" interval cover 1.0?
TRIALS = 200
hit = 0
ests, ses, kept, confounders = [], [], [], []
for i in range(TRIALS):
    Zi, di, yi = make(4000 + i)
    las_i = LassoCV(cv=5, random_state=RANDOM_STATE, max_iter=20_000).fit(
        np.column_stack([di, Zi]), yi)
    keep_i = np.flatnonzero(np.abs(las_i.coef_[1:]) > 1e-8)
    fit = sm.OLS(yi, sm.add_constant(np.column_stack([di, Zi[:, keep_i]]))).fit()
    lo, hi = fit.conf_int()[1]
    hit += lo <= TRUE_EFFECT <= hi
    ests.append(fit.params[1])
    ses.append(fit.bse[1])
    kept.append(len(keep_i))
    confounders.append(np.sum(keep_i < 5))      # the confounders are columns 0 to 4

p_cov = hit / TRIALS
print(f"naive post-selection coverage : {p_cov:.1%}  "
      f"(MC SE {(p_cov*(1-p_cov)/TRIALS)**0.5:.1%})   nominal 95%")
print(f"mean estimate                 : {np.mean(ests):.4f}   truth {TRUE_EFFECT}")
print(f"mean controls kept            : {np.mean(kept):.1f} of {Z.shape[1]}, "
      f"of which confounders {np.mean(confounders):.1f} of 5")
print(f"SD of the 200 estimates       : {np.std(ests, ddof=1):.4f}   "
      f"mean reported SE {np.mean(ses):.4f}")

**Expected output:** coverage 82.5% (MC SE 2.7%) against a nominal 95%, and a mean estimate of
1.0400 against a truth of 1.0. On average LASSO kept 26.2 controls, and all five confounders in
every sample (5.0 of 5). The 200 estimates spread with a standard deviation of 0.0906, while the
standard error OLS reported averaged 0.0693.

### YOUR DIAGNOSIS

1. Coverage is far below 95%. Explain the mechanism: what did OLS's standard error fail to account for?
2. The mean estimate is off too. Say in which direction, and why choosing the controls on the same data can bias the estimate, not only the interval, when every confounder was kept.
3. A colleague proposes fixing this with a larger penalty, so that fewer controls are kept. Explain why that makes it worse.

*Your diagnosis here:*

---

## Diagnosis 2: Double Selection (30 min)

Belloni, Chernozhukov and Hansen (2014) replace the analyst's one LASSO with two:
1. LASSO of the outcome `y` on the controls `Z`, without `d`. The controls it keeps are support A.
2. LASSO of the treatment `d` on `Z`. The controls it keeps are support B.
3. OLS of `y` on `d` and every control in A or B (the union), with a constant. The coefficient on
   `d`, its standard error and its 95% interval are the result.

The next cell gives you steps 1 and 2 as one function: `lasso_support(X, y)` runs the analyst's
`LassoCV` of `y` on the columns of `X` and returns the positions of the columns it keeps (the
*support*), never the coefficients. `seed=RANDOM_STATE` gives `seed` a default value, so
`lasso_support(X, y)` uses `RANDOM_STATE`.

In [ ]:
# GIVEN — run as-is
# Steps 1 and 2 of double selection: the positions of the columns LASSO keeps
def lasso_support(X, y, seed=RANDOM_STATE):
    """The positions of the columns LASSO keeps, with the analyst's settings. Not the coefficients."""
    fit = LassoCV(cv=5, random_state=seed, max_iter=20_000).fit(X, y)
    return np.flatnonzero(np.abs(fit.coef_) > 1e-8)

**Your task.** In the empty cell below:
1. Write a function `double_selection(Z, d, y, seed=RANDOM_STATE)` that does the three steps,
   calling `lasso_support` with `seed` for steps 1 and 2. It returns a dictionary with four
   entries: `"estimate"`, the coefficient on `d`; `"se"`, its standard error; `"ci"`, its 95%
   interval as a pair `(lo, hi)`; and `"n_union"`, the number of controls in the union.
2. Under the function, print `double_selection(Z, d, y)`.

**Reading for the task.** `set(a)` turns an array of positions into a set; `A | B` is the union
of two sets, everything in either, and `len(A)` counts a set's members. `sorted(...)` turns a set
back into a list in increasing order, which `Z[:, ...]` can use. A fitted OLS's `.bse` holds the
standard errors, in the same order as `.params`. `{"estimate": b, "se": s}` builds a dictionary
from names and values. The OLS lines of the analyst's cell in Diagnosis 1 are the closest model;
the function takes about four lines.

In [ ]:
# YOUR TASK — Diagnosis 2: write double_selection, then print double_selection(Z, d, y)

**Expected output:** `{'estimate': 0.9817..., 'se': 0.0721..., 'ci': (0.8394..., 1.1240...),
'n_union': 15}`, with more decimals. If a number prints as `np.float64(0.98...)`, that is numpy's
way of writing a number, and it is fine.

The next cell runs your function on Diagnosis 1's 200 samples, the same seeds, and on each sample
also fits OLS on the five true controls only, which only a simulation can do. It prints a table
with the analyst's row from Diagnosis 1 beside them. `globals()` holds every name defined so far
in this notebook, so the cell can tell whether `double_selection` exists yet. It takes about
fifteen seconds.

In [ ]:
# GIVEN — run as-is, once double_selection is written
# Your double_selection and OLS on the five true controls, on Diagnosis 1's 200 samples
if "double_selection" not in globals():
    print("Not written yet: write double_selection in the cell above, then run this cell again.")
else:
    def row(name, cover, est, se, n_kept):
        """One line of the table: coverage, its MC SE, and a summary of the 200 estimates."""
        print(f"{name:<23}{cover:>9.1%}{(cover * (1 - cover) / TRIALS) ** 0.5:>7.1%}"
              f"{np.mean(est):>10.4f}{np.std(est, ddof=1):>11.4f}{np.mean(se):>9.4f}"
              f"{np.mean(n_kept):>10.1f}")

    results = {"double selection": [], "true controls only": []}
    for i in range(TRIALS):
        Zi, di, yi = make(4000 + i)
        result = double_selection(Zi, di, yi)
        results["double selection"].append(
            (result["estimate"], result["se"], *result["ci"], result["n_union"]))
        true_fit = sm.OLS(yi, sm.add_constant(np.column_stack([di, Zi[:, :5]]))).fit()
        results["true controls only"].append(
            (true_fit.params[1], true_fit.bse[1], *true_fit.conf_int()[1], 5))

    print(f"{'':<23}{'coverage':>9}{'MC SE':>7}{'mean est':>10}{'SD of est':>11}"
          f"{'mean SE':>9}{'controls':>10}")
    for name, values in results.items():
        est, se, lo, hi, n_kept = np.array(values).T
        row(name, np.mean((lo <= TRUE_EFFECT) & (TRUE_EFFECT <= hi)), est, se, n_kept)
    row("analyst (Diagnosis 1)", p_cov, ests, ses, kept)

**Reading the cell.** For each procedure, `results` collects one tuple per sample: the
estimate, its standard error, the interval's two ends (`*result["ci"]` unpacks the pair into two
entries) and the number of controls. `np.array(values).T` turns the 200 tuples into five columns,
which `est, se, lo, hi, n_kept = ...` names in order, and `&` is "and" for arrays of True/False.
`row` prints one line of the table; the analyst's line reuses Diagnosis 1's lists.

**Expected output:**

| | coverage | MC SE | mean est | SD of est | mean SE | controls |
|---|---|---|---|---|---|---|
| double selection | 88.5% | 2.3% | 0.9564 | 0.0957 | 0.0812 | 26.5 |
| true controls only | 95.0% | 1.5% | 0.9987 | 0.0785 | 0.0726 | 5.0 |
| analyst (Diagnosis 1) | 82.5% | 2.7% | 1.0400 | 0.0906 | 0.0693 | 26.2 |

A function that follows the task gets these numbers exactly. None of the three procedures drops a
confounder on this design: Diagnosis 1 printed that the analyst kept all five every time.

### YOUR DIAGNOSIS

1. Double selection covers more often than the analyst's procedure, yet its mean estimate is about as far from 1.0, on the other side, and it still falls short of the 95.0% that OLS on the true controls reaches. Use the "SD of est" and "mean SE" columns, and the number of controls, to explain where each procedure's shortfall comes from, and what choosing about 21 extra controls on the same data does to the estimate and to the standard error OLS reports.

*Your diagnosis here:*

---

## Diagnosis 3: The Shrunk Coefficient (20 min)

A colleague fits LASSO and quotes one of its coefficients in a policy memo as an elasticity. To see
what that number is, use a design where every true coefficient is known: 400 rows, 60 predictors
drawn as standard normal numbers, and only the first eight matter, with true coefficients from 3.0
down to −0.6.

**Reading the code.** `np.zeros(p_s)` is 60 zeros, and `beta_true[:8] = [...]` sets the first
eight. `set(...)` makes a set of positions; `&` keeps the positions in both sets, and `-` those in
the first set only. `.alpha_` is the λ that `LassoCV` chose: a trailing underscore marks something
learned during `.fit()`. In the second cell, `ratio` is LASSO's coefficient as a share of the true
one, both without their signs, so `1 - ratio` is how much LASSO shrank it. `{x:>9.3f}`
right-aligns x in 9 characters with 3 decimals, and `{x:>11.1%}` prints a share as a percentage.

In [ ]:
# GIVEN — run as-is
# LASSO on a design where every true coefficient is known
r_shrink = np.random.default_rng(7)
n_s, p_s = 400, 60
X_s = r_shrink.normal(0, 1, (n_s, p_s))
beta_true = np.zeros(p_s)
beta_true[:8] = [3.0, -2.5, 2.0, -1.5, 1.2, -1.0, 0.8, -0.6]     # only the first eight matter
y_s = X_s @ beta_true + r_shrink.normal(0, 1.0, n_s)

las_s = LassoCV(cv=5, random_state=RANDOM_STATE, max_iter=20_000).fit(X_s, y_s)
kept = np.flatnonzero(np.abs(las_s.coef_) > 1e-8)
true_support = set(np.flatnonzero(beta_true))
print(f"LassoCV chose lambda = {las_s.alpha_:.4f}")
print(f"kept {len(kept)} of {p_s}; {len(set(kept) & true_support)} of "
      f"{len(true_support)} true, {len(set(kept) - true_support)} false positives")

In [ ]:
# GIVEN — run as-is
# Each true coefficient beside LASSO's, and how much LASSO shrank it
print(f"{'var':>5}{'true':>9}{'LASSO':>9}{'shrunk by':>12}")
print("-" * 35)
ratios = []
for j in range(8):
    ratio = abs(las_s.coef_[j]) / abs(beta_true[j])
    ratios.append(ratio)
    print(f"{j:>5}{beta_true[j]:>9.3f}{las_s.coef_[j]:>9.3f}{1 - ratio:>11.1%}")

print(f"\nmean shrinkage among kept true variables: {1 - np.mean(ratios):.1%}")

**Expected output:** `LassoCV` chose λ = 0.0519 and kept 19 of the 60 predictors: all 8 true
ones and 11 false positives. LASSO's coefficients are 2.958, −2.492, 1.941, −1.446, 1.190,
−0.984, 0.678 and −0.534, shrunk by 1.4%, 0.3%, 3.0%, 3.6%, 0.9%, 1.6%, 15.2% and 11.1%; the mean
shrinkage is 4.6%.

**Post-LASSO** is the usual repair when you need sizes: OLS on the variables LASSO kept, with no
penalty. `kept` lists positions in increasing order, and the eight true variables, 0 to 7, are all
in it and come first, so after the constant the coefficient on variable j sits at position j + 1
of `post.params`.

In [ ]:
# GIVEN — run as-is
# Post-LASSO: OLS on the variables LASSO kept, beside the truth and LASSO
post = sm.OLS(y_s, sm.add_constant(X_s[:, kept])).fit()
print(f"{'var':>5}{'true':>9}{'LASSO':>9}{'post-LASSO OLS':>16}")
print("-" * 39)
for j in range(8):
    print(f"{j:>5}{beta_true[j]:>9.3f}{las_s.coef_[j]:>9.3f}{post.params[j + 1]:>16.3f}")

**Expected output:** post-LASSO gives 3.013, −2.553, 1.984, −1.481, 1.248, −1.052, 0.742 and
−0.588.

### YOUR DIAGNOSIS

1. Read the "shrunk by" column. For which coefficients is the shrinkage largest, and why does that matter for the coefficients a policy memo is most likely to quote?

*Your diagnosis here:*

### Diagnosis 3, continued: λ chosen in the open

How far LASSO shrinks is set by λ, and `LassoCV` chose λ out of sight: 100 penalties of its own,
the rows dealt into 5 folds in order, and no scaler. No scaler was nearly harmless above, because
every column of `X_s` is on the same scale. Real data is not. The next two cells store the same 60
predictors with every third one recorded in dollars rather than thousands of dollars, so those
columns are 1,000 times larger. The penalty charges by the size of a coefficient, and a column
1,000 times larger needs a coefficient 1,000 times smaller, so without a scaler LASSO would barely
penalise those columns. The rule from class: standardise, with the scaler fitted on each fold's
training rows only, so the held-out rows play no part in it.

**Reading the code.**
- `np.arange(p_s) % 3 == 0` is True for columns 0, 3, 6, ... (`%` is the remainder), and
  `np.where(cond, 1000, 1)` gives 1000 where it is True and 1 elsewhere. `X_s * units` multiplies
  each column by its entry. `pd.DataFrame(..., columns=[...])` makes a table with columns `x0` to
  `x59`, and `design["y"] = y_s` adds the outcome.
- `LAMBDAS = np.logspace(-3, 0, 31)` is 31 penalties from 10⁻³ = 0.001 to 10⁰ = 1, evenly spaced
  in powers of ten.
- `folds` deals the rows into 5 folds after shuffling them with a fixed seed. `folds.split(X)`
  gives one pair of arrays for each fold: the positions of its training rows and of its held-out
  rows.
- `StandardScaler()` rescales each column: it subtracts the column's mean and divides by its
  standard deviation, both learned when it is fitted.
- `make_pipeline(StandardScaler(), Lasso())` chains the two steps: `.fit` fits the scaler, then
  fits LASSO on the scaled columns; `.predict` applies the fitted scaler first, then LASSO.
- `GridSearchCV(model, {"lasso__alpha": LAMBDAS}, cv=folds, ...)` runs the whole search in one
  call: for each penalty and each of the 5 folds, it fits a new copy of the pipeline, scaler
  included, on the training rows and scores it on the held-out rows. A pipeline names each step
  after its class in lower case, so `lasso__alpha` (two underscores) is the `alpha` of the step
  called `lasso`.
- scikit-learn keeps the *largest* score, so to minimise the mean squared error it scores by minus
  the MSE, `"neg_mean_squared_error"`; the minus signs below turn it back.
  `.cv_results_["mean_test_score"]` holds the score for each penalty, averaged over the 5 folds,
  in grid order. `.best_params_` and `.best_score_` are the winner and its score.
- `design.drop(columns="y")` is every column except `y`.

In [ ]:
# GIVEN — run as-is
# Diagnosis 3's predictors, with every third column recorded in dollars (x 1,000)
units = np.where(np.arange(p_s) % 3 == 0, 1000, 1)
design = pd.DataFrame(X_s * units, columns=[f"x{j}" for j in range(p_s)])
design["y"] = y_s

In [ ]:
# GIVEN — run as-is
# lambda chosen by 5-fold CV over a grid you set, with the scaler fitted inside every fold
LAMBDAS = np.logspace(-3, 0, 31)     # 31 penalties from 0.001 to 1
folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = GridSearchCV(make_pipeline(StandardScaler(), Lasso()), {"lasso__alpha": LAMBDAS},
                      cv=folds, scoring="neg_mean_squared_error")
search.fit(design.drop(columns="y"), design["y"])

cv_mse = -search.cv_results_["mean_test_score"]     # one CV MSE per penalty
lam_cv = search.best_params_["lasso__alpha"]
cv_best = -search.best_score_
print(f"CV chooses lambda = {lam_cv:.4f}, with a CV MSE of {cv_best:.4f}")

**Expected output:** "CV chooses lambda = 0.0501, with a CV MSE of 1.0173". It is close to
`LassoCV`'s 0.0519, which is not on this grid. Write it yourself, next, rebuilds both numbers
with a loop you finish.

---

## Write it yourself (required — 25 min)

This function is required. No AI: write it yourself.

`GridSearchCV` ran the cross-validation out of sight. In the cell below, finish
`choose_lambda(df, outcome, lambdas)`, which runs the same search as a loop and returns two
numbers, in this order: the chosen λ and its CV MSE. `df` is a table, `outcome` is the name of
its outcome column, and every other column of `df` is a predictor. The outer loop over the
penalties, the mean over the folds, the choice of the smallest CV MSE and the return are given.

Replace the `____` with the inner loop, for one penalty `lam`: for each of the 5 folds of
`folds`, fit a new pipeline, `StandardScaler()` then `Lasso(alpha=lam)`, on the fold's training
rows, and append its mean squared error on the fold's held-out rows to `fold_mse`. Use `Lasso`,
not `GridSearchCV` or `LassoCV`.

**Tools.** `folds.split(X)` gives the row *positions* of each fold's training and held-out rows,
so pick rows with `.iloc[...]`, which selects by position. `metrics.mean_squared_error(y_true,
y_pred)` is the MSE. You may instead use `cross_val_score(model, X, y, cv=folds,
scoring="neg_mean_squared_error")`, which fits a new copy of `model` on each fold's training rows
and gives back the 5 scores, each one minus an MSE.

You need about four lines. The check runs it on `design` with Diagnosis 3's 31 penalties, where it
should give λ = 0.0501 and a CV MSE of 1.0173, in every digit; with every second penalty, where it
should match Diagnosis 3's curve at those penalties; and on every second row, with the columns
renamed, against `GridSearchCV`. Many wrong loops choose the same λ, 0.0501: the CV MSE is what
tells them apart. Until you replace the `____`, the check says "Not written yet"; once you do, it
fails with a message until your function is right.

In [ ]:
# GIVEN — run as-is: metrics.mean_squared_error, which choose_lambda() uses on each held-out fold
# It is the average squared gap between actual and predicted, written as a function call. The
# hand formula gives the same number; here both are on the grid search's best model, in sample.
demo_pred = search.best_estimator_.predict(design.drop(columns="y"))   # the X the search was fitted on
print(f"by hand:              {np.mean((design['y'] - demo_pred) ** 2):.4f}")
print(f"mean_squared_error:   {metrics.mean_squared_error(design['y'], demo_pred):.4f}")

In [ ]:
# WRITE IT YOURSELF — required: finish choose_lambda (the inner loop over the folds)
def choose_lambda(df, outcome, lambdas):
    """lambda chosen by 5-fold CV over lambdas, with the scaler fitted inside every fold."""
    X = df.drop(columns=outcome)
    y = df[outcome]
    curve = []                       # one CV MSE per penalty, in the order of lambdas
    for lam in lambdas:
        fold_mse = []                # this penalty's 5 fold MSEs
        # Your lines: for each fold in folds, fit a new pipeline and append its MSE to fold_mse
        ____
        curve.append(np.mean(fold_mse))
    best = int(np.argmin(curve))     # the position of the smallest CV MSE
    return lambdas[best], curve[best]

In [ ]:
# CHECK — for choose_lambda above. Run it; you need not read it.
# globals() holds every name defined so far in this notebook
def names_used(code):
    """Every global name a function's code refers to, including inside functions it defines."""
    names = set(code.co_names)
    for const in code.co_consts:
        if hasattr(const, "co_names"):
            names |= names_used(const)
    return names


BLANK = "_" * 4                  # four underscores, the blank in the given cell

if "choose_lambda" not in globals() or BLANK in names_used(choose_lambda.__code__):
    print(f"Not written yet: replace the {BLANK} in choose_lambda above, then run this check again.")
else:
    import copy
    import inspect

    def same(a, b):
        """Equal to about nine significant digits: only rounding inside the computer passes."""
        return bool(np.isclose(a, b, rtol=1e-9, atol=0))

    def near(a, b):
        """Equal to about six significant digits: for the known slips below."""
        return bool(np.isclose(a, b, rtol=1e-6, atol=0))

    def rounded(a, b):
        """True if a is b rounded to some number of decimals, but not b itself."""
        return not same(a, b) and any(a == round(b, k) for k in range(7))

    # inspect.signature lists the names of a function's arguments, in order
    params = list(inspect.signature(choose_lambda).parameters)
    assert params[:3] == ["df", "outcome", "lambdas"], (
        f"choose_lambda's arguments are ({', '.join(params)}). The check calls it as the task "
        "asks: choose_lambda(df, outcome, lambdas).")
    used = names_used(choose_lambda.__code__)
    for tool in ["GridSearchCV", "LassoCV", "ElasticNetCV"]:
        assert tool not in used, (
            f"choose_lambda calls {tool}. Write the loop over the penalties yourself.")
    assert "search" not in used, (
        "choose_lambda uses search, Diagnosis 3's fitted GridSearchCV. Fit new pipelines inside "
        "the function.")
    assert "design" not in used, (
        "choose_lambda uses design, the lab's table, instead of its df argument.")
    assert "LAMBDAS" not in used, (
        "choose_lambda uses LAMBDAS instead of its lambdas argument.")

    # Diagnosis 3's numbers, read from search itself, so a name you reuse cannot change them
    try:
        grid = np.asarray(search.param_grid["lasso__alpha"])
        curve = -search.cv_results_["mean_test_score"]      # one CV MSE per penalty
        lam_target = float(search.best_params_["lasso__alpha"])
        mse_target = float(-search.best_score_)
        lab_folds = search.cv
        search.best_estimator_[-1].coef_
    except Exception:
        raise AssertionError(
            "search is no longer Diagnosis 3's fitted GridSearchCV: a cell gave that name to "
            "something else. Give your own results other names, then rerun the notebook from "
            "the top.") from None

    # Diagnosis 3's objects, as Diagnosis 3 left them
    rebuilt = pd.DataFrame(X_s * units, columns=[f"x{j}" for j in range(p_s)])
    rebuilt["y"] = y_s
    assert design.equals(rebuilt), (
        "design is not the table Diagnosis 3 made. Does choose_lambda change df in place "
        "(inplace=True, .pop, or df[...] = ...)? Make it leave df as it is, then rerun the "
        "notebook from the top.")
    refit = make_pipeline(StandardScaler(), Lasso(alpha=lam_target)).fit(
        design.drop(columns="y"), design["y"])
    assert np.allclose(search.best_estimator_[-1].coef_, refit[-1].coef_), (
        "search no longer holds the model Diagnosis 3 fitted: something refitted it. Fit a new "
        "pipeline inside choose_lambda for every penalty and fold, then rerun the notebook from "
        "the top.")
    def coefs(model):
        """A fitted model's coefficients: for search, those of the model it refitted."""
        model = getattr(model, "best_estimator_", model)
        return model[-1].coef_ if hasattr(model, "steps") else model.coef_

    # deep copies of the lab's fitted models, put back if choose_lambda refits one
    protected = {"search": copy.deepcopy(search), "las_s": copy.deepcopy(las_s)}
    refitted = set()

    def shown(value):
        """A short description of what choose_lambda gave back."""
        if isinstance(value, dict):
            return "a dictionary with " + ", ".join(repr(k) for k in value)
        return f"a {type(value).__name__}"

    def run(where, df, outcome, lambdas):
        """Call choose_lambda on copies; return its two numbers, or stop with a plain message."""
        try:
            got = choose_lambda(copy.deepcopy(df), outcome, lambdas.copy())
        except KeyError as err:
            full = " ".join(str(err).split())            # the message on one line
            text = full if len(full) < 70 else full[:60] + " ..."
            if "not in index" in full or full.strip("'\"").lstrip("-").isdigit():
                hint = (" Are rows picked with .loc, or with y[...] in plain brackets? "
                        "folds.split gives row positions, which .iloc uses; .loc and plain "
                        "brackets look rows up by their labels, and here the labels are "
                        "0, 2, 4, ...")
            elif outcome != "y":
                hint = (" Does it use its outcome argument, and the columns of df, or the lab's "
                        "column names?")
            else:
                hint = ""
            raise AssertionError(f"{where}, choose_lambda stopped with KeyError {text}.{hint}") from None
        except Exception as err:
            raise AssertionError(f"{where}, choose_lambda stopped with {type(err).__name__}: "
                                 f"{str(err).rstrip('.')}.") from None
        finally:
            for name, saved in protected.items():
                if not np.allclose(coefs(globals()[name]), coefs(saved)):
                    globals()[name] = copy.deepcopy(saved)     # put the lab's model back
                    refitted.add(name)
        assert got is not None, "choose_lambda gives back nothing: use return, not print."
        assert not hasattr(got, "fit"), (
            "choose_lambda returns a model. Return two numbers: the chosen lambda, then its CV MSE.")
        try:
            lam, mse = got                    # fails unless there are exactly two things
            return float(lam), float(mse)
        except (TypeError, ValueError):
            raise AssertionError(f"choose_lambda gives back {shown(got)}, not two numbers. Return "
                                 "the chosen lambda, then its CV MSE.") from None

    # 1. design, with Diagnosis 3's 31 penalties
    where = "On design with Diagnosis 3's 31 penalties"
    lam1, mse1 = run(where, design, "y", grid)
    assert not (same(lam1, mse_target) and same(mse1, lam_target)), (
        "choose_lambda gives the CV MSE first and lambda second. Return lambda first.")
    assert not (rounded(lam1, lam_target) or rounded(mse1, mse_target)), (
        f"choose_lambda gives {lam1} and {mse1}: is a number rounded? Return them unrounded.")
    assert np.isclose(lam1, grid, rtol=1e-9, atol=0).any(), (
        f"{where}, choose_lambda chooses lambda = {lam1:.4g}, which is not one of the penalties "
        "it was given. Does it loop over its lambdas argument, or over penalties of its own?")
    assert mse1 > 0, (
        f"{where}, choose_lambda gives a CV MSE of {mse1:.4f}, below zero. A score of "
        "\"neg_mean_squared_error\" is minus the MSE.")
    SLIPS = {
        (0.0501187234, 1.0380585651): "come from a model with no scaler, so the columns recorded in "
                                      "dollars are barely penalised. Standardise inside every fold",
        (0.0501187234, 1.0179866452): "come from scaling every row before the folds are dealt, so the "
                                      "held-out rows helped set each fold's scaler. Fit the scaler on "
                                      "each fold's training rows only",
        (0.0501187234, 1.0179818242): "come from LassoCV inside a pipeline: the pipeline fits the scaler "
                                      "once, on every row, before LassoCV deals its folds",
        (0.0794328235, 1.0444803424): "pick the largest penalty within one standard error of the best, "
                                      "the one-standard-error rule. The task asks for the penalty with "
                                      "the smallest CV MSE",
        (1.0, 7.8464650097): "pick the largest penalty on the grid, where the CV MSE is the largest of "
                             "all. Does it take argmin of minus the MSE, or pass 1 / lambda to Lasso? "
                             "scikit-learn's alpha is lambda itself",
        (0.0398107171, 1.0872203199): "come from ElasticNet, which adds Ridge's penalty to LASSO's. "
                                      "Fit Lasso",
        (0.316227766, 7.5153572182): "come from ElasticNet with the penalties as its l1_ratio, the mix "
                                     "of LASSO and Ridge that glmnet calls alpha. scikit-learn's alpha "
                                     "is lambda itself",
        (0.0501187234, 0.9980374035): "come from folds dealt in order, without shuffling. Use the "
                                      "lab's folds",
        (0.0501187234, 1.0282704553): "come from folds shuffled with another seed. Use the lab's folds",
        (0.0501187234, 0.9955538253): "come from 10 folds. Use the lab's folds, which are 5",
        (0.0501187234, 1.2467138926): "come from scaling the held-out rows with their own mean and "
                                      "standard deviation. Scale them with the scaler fitted on the "
                                      "training rows",
        (0.0501187234, 1.0070532432): "average the folds' root mean squared errors. Average the MSEs",
        (0.0501187234, 5.0867110389): "add up the 5 fold MSEs. A penalty's CV MSE is their mean",
        (0.0501187234, 1.0313957083): "take the median of the 5 fold MSEs. A penalty's CV MSE is "
                                      "their mean",
        (0.001, 0.7620941244): "come from scoring each model on the rows it was fitted on. Score it on the fold's "
                               "held-out rows",
        (0.0501187234, 0.9599603469): "come from R-squared, which cross_val_score reports when no "
                                      "scoring is given. Score by the mean squared error",
        (0.001, 1.0125680609e-06): "come from a model with the outcome among its predictors. Use every "
                                   "column except outcome",
        (1.0, 4262137.2733): "come from a model fitted on scaled training rows and then asked to "
                             "predict from unscaled held-out rows. Pass the held-out rows through "
                             "the scaler fitted on the training rows",
        (1.0, 0.6983439577): "come from R-squared, where larger is better, so the smallest picks the "
                             "worst penalty. Score by the mean squared error",
    }
    for (lam_slip, mse_slip), cause in SLIPS.items():
        assert not (near(lam1, lam_slip) and near(mse1, mse_slip)), (
            f"{where}, choose_lambda gives lambda = {lam1:.4f} and CV MSE {mse1:.6f}; "
            f"GridSearchCV gave {lam_target:.4f} and {mse_target:.6f}. Your numbers {cause}.")
    if same(lam1, lam_target) and not same(mse1, mse_target) and abs(mse1 / mse_target - 1) < 1e-4:
        raise AssertionError(
            f"{where}, your lambda matches and your CV MSE is {100 * (mse1 / mse_target - 1):+.2g}% "
            "away. Lasso stops once it is close enough to the exact answer, so a fit that starts "
            "from the previous one (warm_start=True, or lasso_path) lands slightly apart. Fit a "
            "new pipeline for every penalty and fold.")
    again = run(where, design, "y", grid)
    assert again == (lam1, mse1), (
        "choose_lambda gives a different answer each time it runs. Are the rows shuffled "
        "without a fixed seed? Use the lab's folds.")
    assert same(lam1, lam_target) and same(mse1, mse_target), (
        f"{where}, choose_lambda gives lambda = {lam1:.4f} and CV MSE {mse1:.6f}; GridSearchCV "
        f"gave {lam_target:.4f} and {mse_target:.6f}. Compare each step with the task: the "
        "folds, the scaler and the rows it is fitted on, the model, the score, and how the 5 "
        "scores are combined.")

    # 2. every second penalty: Diagnosis 3's curve at those penalties
    half_grid = grid[::2]
    best = int(np.argmin(curve[::2]))
    lam2, mse2 = run("With every second penalty", design, "y", half_grid)
    assert not (same(lam2, lam_target) and same(mse2, mse_target)), (
        "With every second penalty, choose_lambda gives the answer for all 31. Does it loop over "
        "its lambdas argument?")
    assert same(lam2, half_grid[best]) and same(mse2, curve[::2][best]), (
        f"With every second penalty, choose_lambda gives lambda = {lam2:.4f} and CV MSE "
        f"{mse2:.6f}; Diagnosis 3's curve has its smallest CV MSE among those penalties at "
        f"{half_grid[best]:.4f}, {curve[::2][best]:.6f}.")

    # 3. every second row, with the columns renamed and the outcome first
    names = {**{f"x{j}": f"inc_{j}" for j in range(p_s)}, "y": "spend"}
    other = design.iloc[::2].rename(columns=names)
    other = other[["spend"] + [f"inc_{j}" for j in range(p_s)]]
    where = "On every second row, with the columns renamed (spend, inc_0 to inc_59)"
    want = GridSearchCV(make_pipeline(StandardScaler(), Lasso()), {"lasso__alpha": grid},
                        cv=lab_folds, scoring="neg_mean_squared_error")
    want.fit(other.drop(columns="spend"), other["spend"])
    lam3, mse3 = run(where, other, "spend", grid)
    assert not (same(lam3, lam_target) and same(mse3, mse_target)), (
        f"{where}, choose_lambda gives the answer for design. Does it use its df argument?")
    assert same(lam3, want.best_params_["lasso__alpha"]) and same(mse3, -want.best_score_), (
        f"{where}, choose_lambda gives lambda = {lam3:.4f} and CV MSE {mse3:.6f}; GridSearchCV "
        f"gives {want.best_params_['lasso__alpha']:.4f} and {-want.best_score_:.6f}. Are the "
        "predictors every column except outcome, wherever outcome sits, and does every line use "
        "df rather than the lab's table?")

    assert not refitted, (
        f"choose_lambda refits {' and '.join(sorted(refitted))}, which Diagnosis 3 fitted. The "
        "check has put it back; fit a new pipeline inside the function instead.")

    print(f"choose_lambda matches Diagnosis 3's GridSearchCV: lambda = {lam1:.4f}, "
          f"CV MSE {mse1:.4f}.")
    print(f"With every second penalty: lambda = {lam2:.4f}, CV MSE {mse2:.4f}.")
    print(f"On every second row, with the columns renamed: lambda = {lam3:.4f}, "
          f"CV MSE {mse3:.4f}.")

---

## Ship It: `src/sparse_tools.py` (10 min)

Here is the module, finished except for one function. Its design rule is the one from class:
`lasso_support` returns a *support*, the positions of the columns LASSO kept, never a coefficient
vector that someone could quote as effects. The causal estimate comes from `double_selection`,
the only function here that returns an interval, and `coverage_study` measures any estimator's
coverage with its Monte Carlo standard error.

**Your task.** Paste your `double_selection` from Diagnosis 2, unchanged, at the end of the first
cell, where it says so, then run the two cells and compare the output with the expected output
below. The first saves the module as a file, the second imports it and runs it on Diagnosis 1's
samples. Your function's default `seed=RANDOM_STATE` works inside the module, because the module
sets `RANDOM_STATE` itself.

**Reading the cells.** `%%writefile src/sparse_tools.py` saves the cell as that file instead of
running it. The notes after a colon or `->` (`seed: int`, `-> dict`) only say what type each
argument and result is. `coverage_study` takes two functions as arguments, `make_fn` and
`estimator`, and calls them like any other function; `*make_fn(...)` unpacks the three things
`make_fn` returns into three arguments of `estimator`. In the second cell, `import sparse_tools`
loads the file as a module, and `hasattr(sparse_tools, "double_selection")` asks whether the
module defines that name.

In [ ]:
%%writefile src/sparse_tools.py
"""High-dimensional estimation that will not hand you a causal number by accident (ECON 5200 Lab 16).

The design rule: lasso_support returns a support, never coefficients, because the most common
failure in applied high-dimensional work is reporting a shrunk LASSO coefficient as an effect.
Causal estimates come from double_selection, the only function here that returns an interval.
"""
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LassoCV

RANDOM_STATE = 42          # the seed the lab uses, and the default of every function here


def lasso_support(X, y, seed: int = RANDOM_STATE) -> np.ndarray:
    """The positions of the columns LASSO keeps. Deliberately not the coefficients."""
    fit = LassoCV(cv=5, random_state=seed, max_iter=20_000).fit(X, y)
    return np.flatnonzero(np.abs(fit.coef_) > 1e-8)


def coverage_study(make_fn, estimator, truth: float, trials: int = 200,
                   first_seed: int = 4000) -> dict:
    """How often estimator's 95% interval covers truth over trials samples, and its MC SE."""
    hits, widths = 0, []
    for i in range(trials):
        lo, hi = estimator(*make_fn(first_seed + i))["ci"]
        hits += lo <= truth <= hi
        widths.append(hi - lo)
    coverage = hits / trials
    return {"coverage": coverage, "mc_se": (coverage * (1 - coverage) / trials) ** 0.5,
            "mean_width": float(np.mean(widths)), "trials": trials}


# Paste your double_selection function from Diagnosis 2 below this line, unchanged.


In [ ]:
# GIVEN — run as-is, once your double_selection is pasted into the module
# Import the module and run it on Diagnosis 1's samples: it should match Diagnosis 2
import importlib
import sys

sys.path.insert(0, "src")          # let Python find modules in the src folder
import sparse_tools
importlib.reload(sparse_tools)     # reload, so a re-run of the %%writefile cell takes effect

Z, d, y = make(RANDOM_STATE)       # Diagnosis 1's sample again, in case a cell reused these names
if not hasattr(sparse_tools, "double_selection"):
    print("Paste your double_selection from Diagnosis 2 into the module cell above first, "
          "then run that cell and this one again.")
else:
    shipped = sparse_tools.double_selection(Z, d, y)
    lo, hi = shipped["ci"]
    print(f"estimate {shipped['estimate']:.4f}, 95% interval [{lo:.4f}, {hi:.4f}], "
          f"{shipped['n_union']} controls")

    study = sparse_tools.coverage_study(make, sparse_tools.double_selection, TRUE_EFFECT)
    print(f"coverage over {study['trials']} samples: {study['coverage']:.1%} "
          f"(MC SE {study['mc_se']:.1%}), mean interval width {study['mean_width']:.4f}")

**Expected output:** an estimate of 0.9817 with a 95% interval of [0.8394, 1.1241] and 15
controls, as your function gave in Diagnosis 2; then coverage of 88.5% (MC SE 2.3%), as in
Diagnosis 2's table, and a mean interval width of 0.3206. It takes about fifteen seconds.

---

## Extension (Optional — 25 min): Sample Splitting

Another repair to try: choose the controls on one half of the sample and estimate on the other
half, so the rows that produce the interval played no part in choosing the controls. On a half
sample of 100 rows, LASSO has less to go on, and it can now drop a confounder.

**The task.** In the cell below, and in new cells under it:
1. Write a function `split_sample(Z, d, y)` that runs the analyst's LASSO (Diagnosis 1) on the
   first 100 rows to choose the controls, then fits OLS of `y` on `d` and those controls on the
   other 100 rows. It returns a dictionary with `"estimate"` and `"ci"`, as `double_selection`
   does.
2. Write a second version that chooses the controls on the first 100 rows by double selection's
   union instead, and estimates on the other 100 in the same way.
3. Measure the coverage and mean interval width of both with `sparse_tools.coverage_study`, on
   the same 200 samples, and answer the questions below.

In [ ]:
# YOUR TASK — optional: sample splitting

### Interpretation Questions

1. Compare the two versions' coverage. On 100 rows, how often does the analyst's LASSO drop a confounder (count it), and how does that explain the difference?
2. Compare their mean interval widths with double selection's 0.3206 in Ship It. Say what sample splitting costs, and how Chapter 24's cross-fitting gets that back.

*Your answers here:*

1.

2.

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Simulated a treatment effect of 1.0 among 100 candidate controls and
  ran the common procedure: LASSO picks the controls, then OLS on the
  survivors. Its "95%" interval covered the truth in [YOUR VALUE] of 200
  samples
* Wrote double selection (Belloni, Chernozhukov and Hansen 2014), which
  also keeps the controls that predict the treatment: coverage
  [YOUR VALUE], with [YOUR VALUE] controls kept on average
* Compared LASSO coefficients with known true ones: the mean shrinkage
  was [YOUR VALUE], and post-LASSO OLS brought the sizes back
* Wrote the loop over the folds of my own cross-validation over the
  penalty, with the scaler fitted inside every fold, and matched
  GridSearchCV: lambda [YOUR VALUE], CV MSE [YOUR VALUE]

**Please write a README.md entry including:**
1. Project Title: Regularization: Selection, Shrinkage and Valid Intervals
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust',
'comprehensive' or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab16-regularization`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab16-regularization`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab16-regularization`, branch `main`,
   commit message `Lab 16 submission`. This pushes the notebook, and the
   notebook's rendered output is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `src/sparse_tools.py` module this lab saves is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.